In [1]:
import numpy as np
import pandas as pd

# Plantilla

Es necesario ajustar las definiciones, las fuentes de los datos y posiblemente definiciones si la ENEMDU tiene una dimensión geográfica y temporal al mismo tiempo

In [2]:
# data = pd.read_stata(r"Z:\harmonized\ECU\ENEMDU\data_arm\ECU_1990m11_BID.dta") # para bases de stata
data = pd.read_stata(r"datos/ECU_2007m12_BID.dta", convert_categoricals=False) # para bases de stata

In [3]:
df, meta = pd.read_stata(r"datos/ECU_2007m12_BID.dta", iterator=True), None
meta = df.variable_labels()
print("\nVariable labels:")
for col, label in meta.items():
    print(f"{col}: {label}")


Variable labels:
region_BID_c: Regiones BID
region_c: division politico-administrativa, provincia
pais_c: Nombre del PaÃ­s
anio_c: Anio de la encuesta
mes_c: Mes de la encuesta
zona_c: Zona del pais
factor_ch: Factor de expansion del hogar
idh_ch: ID del hogar
idp_ci: ID de la persona en el hogar
factor_ci: Factor de expansion del individuo
sexo_ci: Sexo del individuo
edad_ci: Edad del individuo en aÃ±os
relacion_ci: Relacion o parentesco con el jefe del hogar
civil_ci: Estado civil
jefe_ci: Jefe/a de hogar
nconyuges_ch: # de conyuges en el hogar
nhijos_ch: # de hijos en el hogar
notropari_ch: # de otros familiares en el hogar
notronopari_ch: # de no familiares en el hogar
nempdom_ch: # de empleados domesticos
clasehog_ch: Tipo de hogar
nmiembros_ch: # de miembros en el hogar
miembros_ci: =1: es miembro del hogar
nmayor21_ch: # de familiares mayores a 21 anios en el hogar
nmenor21_ch: # de familiares menores a 21 anios en el hogar
nmayor65_ch: # de familiares mayores a 65 anios en el 

## Revisar los datos

- rn - regiones naturales
- area - area
- prov - provincia
- cuidad - ciudad
- zona - zona
- sector - sector
- panelm - panel
- vivienda - vivienda
- hogar - hogar
- fexp - factor de expansión
- p01 - persona
- p03 - edad
- p63 - ingresos - patronos cta. propia
- p66 - ingreso de asalariados y/o empl. domésticos
- p67 - descuentos de asalariados
- p68b - ingreso en especies de asalariados
- p71b - ingreso recibido por transacciones de capital
- p72b - ingreso por jubilación o pensiones
- ingrl - ingresos del trabajo
- condact - condición de actividad

En esta encuesta las variables y metodología cambian bastante, usaremos p66 de acuerdo a las etiquetas de las variables lo más cercano a las anteriores como la variable de ingreso laboral monetario de la actividad principal asalariada para mantener la concordancia con el resto de los años.

En esta encuesta ya no tenemos las variables que indicaban la condición de actividad en cada mes, por lo que tenemos que cambiar la lógica, la variable panelm representa un trimestre, cambiamos la lógica para recuperar los ingresos y los hogares trimestrales de acuerdo a esta variable. 

In [16]:
data[['p63', 'p66', 'p67', 'p68b', 'p71b', 'p72b', 'ingrl']].mean()

p63        487.212341
p66        224.087882
p67         25.405549
p68b        39.882146
p71b       245.483286
p72b       233.980075
ingrl    25538.392878
dtype: float64

Filtramos solo las columnas de interés para alivar el peso en la memoria

In [11]:
len(data)

76922

In [10]:
data['condact'].value_counts()

condact
7    24087
3    19258
8    15859
1    11147
2     4618
5      883
6      734
0      336
Name: count, dtype: int64

In [6]:
data.columns

Index(['region_BID_c', 'region_c', 'pais_c', 'anio_c', 'mes_c', 'zona_c',
       'factor_ch', 'idh_ch', 'idp_ci', 'factor_ci',
       ...
       'tcylmpri_ci', 'ytot_ci', 'rentaimp_ch', 'autocons_ci', 'autocons_ch',
       'des1_ch', 'des2_ch', 'migantiguo5_ci', 'migrantelac_ci', 'cpi'],
      dtype='object', length=419)

In [8]:
data = data[['rn', 'area', 'prov', 'ciudad', 'zona', 'sector', 'panelm',
             'vivienda', 'hogar', 'fexp', 'p01', 'p03', 'p63', 'p66', 
             'p67', 'p68b', 'p71b', 'p72b', 'ingrl', 'condact', 
              'ene', 'feb', 'mar', 'abr', 'may', 'jun', 'jul', 'ago', 'sep', 
              'oct', 'nov', 'dic']]

KeyError: "['ene', 'feb', 'mar', 'abr', 'may', 'jun', 'jul', 'ago', 'sep', 'oct', 'nov', 'dic'] not in index"

Por alguna razón extraña tenemos varias columnas del factos de expansión

In [11]:
data['fexp']

,fexp,fexp
0,194.911911,194.911911
1,194.911911,194.911911
2,194.911911,194.911911
3,194.911911,194.911911
4,194.911911,194.911911
...,...,...
77959,230.986725,230.986725
77960,230.986725,230.986725
77961,230.986725,230.986725
77962,230.986725,230.986725


In [12]:
data = data.loc[:, ~data.columns.duplicated()]

In [13]:
data.columns

Index(['rn', 'area', 'ciudad', 'zona', 'prov', 'sector', 'panelm', 'vivienda',
       'hogar', 'persona', 'edad', 'fexp', 'pe61', 'pe63', 'pe64', 'pe65b',
       'pe68a', 'pe69b', 'ingrl', 'ene', 'feb', 'mar', 'abr', 'may', 'jun',
       'jul', 'ago', 'sep', 'oct', 'nov', 'dic'],
      dtype='object')

Creamos una variable de ingreso laboral que es igual al ingreso por asalariado

In [14]:
data['ingr'] = data['pe63']

Ingreso mensual asumiendo que las personas reciben el mismo valor reportado en 'ingr' siempre que reportan estar ocupados en un mes, ahora las variables categoricas por mes tienen diferentes leyendas y no aparecen las etiquetas en la base de 2004, planteamos estas etiquetas basandonos en la continuidad más lógica desde diciembre de 2002

En diciembre de 2002 las personas reportadas como Trabajando fueron 6197, en enero de 2004 son 35209
- 1 - Desocupado
- 2 - Buscando trabajo
- 3 - Trabajando

In [15]:
data['ene'].value_counts()

ene
1.0    37798
3.0    31945
2.0     1018
Name: count, dtype: int64

In [16]:
data['ingr_ene'] = data.apply(lambda x: x['ingr'] if x['ene'] == 3 else None, axis=1)
data['ingr_feb'] = data.apply(lambda x: x['ingr'] if x['feb'] == 3 else None, axis=1)
data['ingr_mar'] = data.apply(lambda x: x['ingr'] if x['mar'] == 3 else None, axis=1)
data['ingr_abr'] = data.apply(lambda x: x['ingr'] if x['abr'] == 3 else None, axis=1)
data['ingr_may'] = data.apply(lambda x: x['ingr'] if x['may'] == 3 else None, axis=1)
data['ingr_jun'] = data.apply(lambda x: x['ingr'] if x['jun'] == 3 else None, axis=1)
data['ingr_jul'] = data.apply(lambda x: x['ingr'] if x['jul'] == 3 else None, axis=1)
data['ingr_ago'] = data.apply(lambda x: x['ingr'] if x['ago'] == 3 else None, axis=1)
data['ingr_sep'] = data.apply(lambda x: x['ingr'] if x['sep'] == 3 else None, axis=1)
data['ingr_oct'] = data.apply(lambda x: x['ingr'] if x['oct'] == 3 else None, axis=1)
data['ingr_nov'] = data.apply(lambda x: x['ingr'] if x['nov'] == 3 else None, axis=1)
data['ingr_dic'] = data.apply(lambda x: x['ingr'] if x['dic'] == 3 else None, axis=1)

In [19]:
data[['ingr_ene', 'ingr_feb', 'ingr_mar', 'ingr_abr', 'ingr_may', 'ingr_jun', 'ingr_jul', 'ingr_ago', 'ingr_sep', 'ingr_oct', 'ingr_nov', 'ingr_dic']].mean()

ingr_ene     52.151013
ingr_feb     47.210762
ingr_mar    107.243860
ingr_abr    107.286074
ingr_may    115.775748
ingr_jun    112.227116
ingr_jul    109.553704
ingr_ago    107.167641
ingr_sep    106.342703
ingr_oct    105.847380
ingr_nov    114.227157
ingr_dic     64.721139
dtype: float64

## Deflactamos y transformamos el ingreso

Esto deja todo en dólares constantes de 2014

In [20]:
# Carga base de datos con ipc
data_externa = pd.read_excel("data_externa.xlsx", sheet_name='datos')

# filtra año de interés
datos_actual = data_externa[data_externa['Año'] == 2006]
datos_base = data_externa[data_externa['Año'] == 2014]

Diccionarios de ipc, desde 2005, tenemos datos del ipc para más provincias

In [21]:
ipc_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Sierra': fila['Sierra'],
        'Costa': fila['Costa'],
        'Guayaquil': fila['Guayaquil'],
        'Esmeraldas': fila['Esmeraldas'],
        'Machala': fila['Machala'],
        'Manta': fila['Manta'],        
        'Quito': fila['Quito'],
        'Loja': fila['Loja'],
        'Cuenca': fila['Cuenca'],
        'Ambato': fila['Ambato']
    }
     for _, fila in datos_actual.iterrows()
     }

ipc_base_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Sierra': fila['Sierra'],
        'Costa': fila['Costa'],
        'Guayaquil': fila['Guayaquil'],
        'Esmeraldas': fila['Esmeraldas'],
        'Machala': fila['Machala'],
        'Manta': fila['Manta'],        
        'Quito': fila['Quito'],
        'Loja': fila['Loja'],
        'Cuenca': fila['Cuenca'],
        'Ambato': fila['Ambato']
    }
     for _, fila in datos_base.iterrows()
     }

### Creamos identificadores para las ciudades siguiendo los códigos del INEC y para los trimestres

In [23]:
# Corregimos los códigos para usarlos cómo texto
data['ciudad'] = data['ciudad'].apply(str)
data['ciudad'] = data['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)

data['ciudad_2'] = data['ciudad'].apply(lambda x: x[:2])

Diccionario ciudades disponibles

In [24]:
parroquia_dict = {
    '01': 'Cuenca',
    '09': 'Guayaquil',
    '17': 'Quito',
    '08': 'Esmeraldas',
    '13': 'Costa',
    '12': 'Costa',
    '07': 'Costa',
    '10': 'Sierra',
    '02': 'Sierra',
    '05': 'Sierra',
    '18': 'Ambato',
    '06': 'Sierra',
    '04': 'Sierra',
    '11': 'Loja',
    '03': 'Sierra'
}

data['ciudad_asignada'] = data['ciudad_2'].apply(lambda x: parroquia_dict.get(x, 'Nacional'))

In [25]:
data['ciudad_asignada'].value_counts()

ciudad_asignada
Sierra        22326
Costa         17593
Guayaquil      9632
Quito          6779
Esmeraldas     6154
Nacional       4220
Loja           3898
Cuenca         3869
Ambato         3493
Name: count, dtype: int64

### Asignamos el ipc correspondiente según ciudad correspondiente

$\begin{equation}
    ingr_{USD-base-2014}^{i} = ingr_{dólares}^{i}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Desde 2000 en adelante ya no es necesario utilizar el tipo de cambio debido al cambio de moneda

In [26]:
# Función que asigna valores correspondientes
def asigna_ipc(fila, trimestre):
    return ipc_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

def asigna_ipc_base(fila, trimestre):
    return ipc_base_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

In [27]:
data['ipc_t1'] = data.apply(lambda fila: asigna_ipc(fila, 1), axis=1)
data['ipc_base_t1'] = data.apply(lambda fila: asigna_ipc_base(fila, 1), axis=1)

data['ipc_t2'] = data.apply(lambda fila: asigna_ipc(fila, 2), axis=1)
data['ipc_base_t2'] = data.apply(lambda fila: asigna_ipc_base(fila, 2), axis=1)

data['ipc_t3'] = data.apply(lambda fila: asigna_ipc(fila, 3), axis=1)
data['ipc_base_t3'] = data.apply(lambda fila: asigna_ipc_base(fila, 3), axis=1)

data['ipc_t4'] = data.apply(lambda fila: asigna_ipc(fila, 4), axis=1)
data['ipc_base_t4'] = data.apply(lambda fila: asigna_ipc_base(fila, 4), axis=1)

In [28]:
# Calculamos el deflactor
data['def_t1'] = (data['ipc_base_t1'] / data['ipc_t1'])
data['def_t2'] = (data['ipc_base_t2'] / data['ipc_t2'])
data['def_t3'] = (data['ipc_base_t3'] / data['ipc_t3'])
data['def_t4'] = (data['ipc_base_t4'] / data['ipc_t4'])

In [29]:
# Ingreso real por mes
data['ingr_ene_r'] = data['ingr_ene'] * data['def_t1']
data['ingr_feb_r'] = data['ingr_feb'] * data['def_t1']
data['ingr_mar_r'] = data['ingr_mar'] * data['def_t1']
data['ingr_abr_r'] = data['ingr_abr'] * data['def_t2']
data['ingr_may_r'] = data['ingr_may'] * data['def_t2']
data['ingr_jun_r'] = data['ingr_jun'] * data['def_t2']
data['ingr_jul_r'] = data['ingr_jul'] * data['def_t3']
data['ingr_ago_r'] = data['ingr_ago'] * data['def_t3']
data['ingr_sep_r'] = data['ingr_sep'] * data['def_t3']
data['ingr_oct_r'] = data['ingr_oct'] * data['def_t4']
data['ingr_nov_r'] = data['ingr_nov'] * data['def_t4']
data['ingr_dic_r'] = data['ingr_dic'] * data['def_t4']

Ingreso mensual promedio en el trimeste

In [30]:
data['ingr_t1_r'] = (data['ingr_ene_r'] + data['ingr_feb_r'] + data['ingr_mar_r'])/3
data['ingr_t2_r'] = (data['ingr_abr_r'] + data['ingr_may_r'] + data['ingr_jun_r'])/3
data['ingr_t3_r'] = (data['ingr_jul_r'] + data['ingr_ago_r'] + data['ingr_sep_r'])/3
data['ingr_t4_r'] = (data['ingr_oct_r'] + data['ingr_nov_r'] + data['ingr_dic_r'])/3

In [31]:
data[['ingr_t1_r', 'ingr_t2_r', 'ingr_t3_r', 'ingr_t4_r']].mean()

ingr_t1_r     70.244068
ingr_t2_r    147.754829
ingr_t3_r    148.046006
ingr_t4_r     88.266275
dtype: float64

## Calculo ingreso de los hogares

En este año no tenemos la variable numpers

In [34]:
columnas_idef = ['rn', 'area', 'prov', 'ciudad', 'zona', 'sector', 'panelm', 'vivienda', 'hogar']

data['idef_hogar'] = data[columnas_idef].astype(str).agg(''.join, axis=1)
len(data['idef_hogar'].unique())

18475

In [35]:
data[['rn', 'area', 'prov', 'ciudad', 'zona', 'sector', 'panelm', 'vivienda',
             'hogar',
      'idef_hogar', 'persona']]

,rn,area,prov,ciudad,zona,sector,panelm,vivienda,hogar,idef_hogar,persona
0,1,1,1,010150,1,6,11,1,1,111010150161111,3
1,1,1,1,010150,1,6,11,1,1,111010150161111,1
2,1,1,1,010150,1,6,11,1,1,111010150161111,2
3,1,1,1,010150,1,6,11,1,1,111010150161111,4
4,1,1,1,010150,1,6,14,1,1,111010150161411,2
...,...,...,...,...,...,...,...,...,...,...,...
77959,2,2,9,091752,930,1,12,1,1,22909175293011211,6
77960,2,2,9,091752,930,1,12,1,1,22909175293011211,3
77961,2,2,9,091752,930,1,12,1,1,22909175293011211,2
77962,2,2,9,091752,930,1,12,1,1,22909175293011211,5


Si todos los miembros del hogar tienen NA como ingreso, mantener NA, si al menos uno tiene un ingreso sumamos para el ingreso del hogar, así evitamos subestimar el ingreso del hogar si tenemos valores perdidos

In [36]:
# Definimos una función que sume pero devuelva NA si todos son NA
def sum_with_na(series):
    if series.isna().all():
        return pd.NA
    else:
        return series.sum(skipna=True)

In [37]:
data['ingr_t1_h'] = data.groupby('idef_hogar')['ingr_t1_r'].transform(sum_with_na)
data['ingr_t2_h'] = data.groupby('idef_hogar')['ingr_t2_r'].transform(sum_with_na)
data['ingr_t3_h'] = data.groupby('idef_hogar')['ingr_t3_r'].transform(sum_with_na)
data['ingr_t4_h'] = data.groupby('idef_hogar')['ingr_t4_r'].transform(sum_with_na)

In [38]:
data[['ingr_t1_h', 'ingr_t2_h', 'ingr_t3_h', 'ingr_t4_h']].mean()

ingr_t1_h     73.907752
ingr_t2_h    137.634081
ingr_t3_h    128.438038
ingr_t4_h    102.896374
dtype: object

In [39]:
print("Ingreso medio de un hogar t4: ", data['ingr_t4_h'].mean())
print("Mediana del ingreso de un hogar t4: ", data['ingr_t4_h'].median())

Ingreso medio de un hogar t4:  102.89637437697968
Mediana del ingreso de un hogar t4:  13.72324744584281


## Sacamos edades negativas y mayores a 100 años

In [40]:
len(data)

77964

En este caso en la variable edad tenemos números y el texto 'menos de un año' así que primero transformamos todas las filas que digan 'menos de un año' a 0

In [41]:
data['edad'] = data['edad'].apply(lambda x: x if type(x) == int else 0)

In [42]:
data = data.loc[(data['edad'] >= 0) & (data['edad'] < 100)]
len(data)

77964

## Ingreso individual descontando cargas familiares

Utilizando la metodología del autor dividimos el ingreso del hogar para la escala $(A_{i}+kC_{i})^{s}$ donde $A_{i}$ es al número de adultos, $C_{i}$ es el número de niños en el hogar $i$. $k$ es el costo en recursos de cada niño y $s$ busca reflejar las restricciones

In [43]:
k = 0.4
s = 0.9

In [44]:
# Si es necesario calcular el número de niños
data['es_nino'] = data['edad'] < 10

data['ninos'] = data.groupby('idef_hogar')['es_nino'].transform('sum')

# Si es necesario calcular el número de adultos
data['es_adulto'] = data['edad'] > 10

data['adultos'] = data.groupby('idef_hogar')['es_adulto'].transform('sum')

In [45]:
data['escala'] = (data['adultos'] + k * data['ninos']) ** s

In [46]:
data['ingr_t_t1'] = data['ingr_t1_h'] / data['escala']
data['ingr_t_t2'] = data['ingr_t2_h'] / data['escala']
data['ingr_t_t3'] = data['ingr_t3_h'] / data['escala']
data['ingr_t_t4'] = data['ingr_t4_h'] / data['escala']

In [47]:
data[['ingr_t_t1', 'ingr_t_t2', 'ingr_t_t3', 'ingr_t_t4']].mean()

ingr_t_t1    19.598238
ingr_t_t2    39.356717
ingr_t_t3    38.679767
ingr_t_t4    24.837593
dtype: object

In [48]:
print("Ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].mean())
print("Mediana del ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].median())

Ingreso individual descontando cargas familiares t4:  24.837592822744032
Mediana del ingreso individual descontando cargas familiares t4:  2.626651375371305


## Umbrales de pobreza

Incluimos los índices de pobreza si es posible a nivel regional para luego poder utilizar de mejor forma el factor de expansión

$\begin{equation}
    umbral_{USD-base-2014}^{i} = umbral_{año}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

Diccionario de umbral

In [49]:
umbral_dict = dict(zip(datos_actual['trimestre'], datos_actual['umbral de pobreza']))

## Cálculo del índice de pobreza de Foster, Greer y Thorbecke

Para calcular un ínidce de pobreza se utiliza a Foster, Greer y Thorbecke (1984), ya que satisface algunas caracterísitcas de distribución que son positivas e igual a las enunciadas por Sen, el autor usa el mismo índice.

$\begin{equation}FGT_{\alpha} = \frac{1}{N}\sum_{i=1}^{H}\left(\frac{z-y_{i}}{z}\right)^{\alpha}\end{equation}$

Donde $z$ es el umbral de pobreza, $N$ es el número de personas en la economía, $H$ es el número de pobres (personas debajo de la línea de pobreza) $y_{i}$ es el ingreso de cada individuo. Mientras mayor es el valor de $\alpha$ mayor es el peso de los individuos más pobres, mayor $FGT$ mayor pobreza en la economía.

En este caso los umbrales están anivel nacional, aún así buscamos calcular la pobreza por región y sacar un promedio ponderado por región para la pobreza nacional, con el objetivo de hacerlo más específico

In [51]:
datos_final = pd.DataFrame(index=['t1', 't2', 't3', 't4'], columns=['fgt0', 'fgt1', 'fgt2', 'a25', 'a50', 'a75', 'ingreso_promedio'])

In [52]:
data['persona_fexp'] = 1 * data['fexp']

In [53]:
for t in [1, 2, 3, 4]:
    col_ingr = f'ingr_t_t{t}'
    col_pobres = f'pobres_t{t}'

    # una columna que identifica a quienes están por debajo de la línea de pobreza por trimestre
    data[col_pobres] = (
        (data[col_ingr] - umbral_dict.get(t)) < 0
    ).astype(int)

In [54]:
print("pobreza t1: ", (data['pobres_t1'] * data['fexp']).sum()/data.loc[data['ingr_t_t1'] >= 0]['persona_fexp'].sum())
print("pobreza t2: ", (data['pobres_t2'] * data['fexp']).sum()/data.loc[data['ingr_t_t2'] >= 0]['persona_fexp'].sum())
print("pobreza t3: ", (data['pobres_t3'] * data['fexp']).sum()/data.loc[data['ingr_t_t3'] >= 0]['persona_fexp'].sum())
print("pobreza t4: ", (data['pobres_t4'] * data['fexp']).sum()/data.loc[data['ingr_t_t4'] >= 0]['persona_fexp'].sum())

pobreza t1:  0.9423979025158848
pobreza t2:  0.8944781586923719
pobreza t3:  0.9038157446383626
pobreza t4:  0.93163715528624


In [55]:
for t in [1, 2, 3, 4]:
    # Filtramos para cada trimestre
    df_temp = data.loc[data[f'ingr_t_t{t}'] >= 0].copy()

    # Calculamos una columna de pobres
    df_temp['pobres'] = (df_temp[f'ingr_t_t{t}'] - umbral_dict[t]) < 0

    # Ratio de pobres sobre el total
    ratio = (umbral_dict[t] - df_temp[f'ingr_t_t{t}']) / umbral_dict[t]

    # Calculamos el índice para alpha 0, 1 y 2 solo donde 'pobres' == True.
    for i in range(3):
        col = f'fgt{i}'
        df_temp[col] = np.where(df_temp['pobres'], ratio**i, 0)

    # Cálculo del índice ponderado: se usa el factor de expansión como peso
    peso_total = df_temp['fexp'].sum()
    fgt0 = (df_temp['fgt0'] * df_temp['fexp']).sum() / peso_total
    fgt1 = (df_temp['fgt1'] * df_temp['fexp']).sum() / peso_total
    fgt2 = (df_temp['fgt2'] * df_temp['fexp']).sum() / peso_total
    
    # Guardamos los resultados
    datos_final.loc[f't{t}', 'fgt0'] = fgt0
    datos_final.loc[f't{t}', 'fgt1'] = fgt1
    datos_final.loc[f't{t}', 'fgt2'] = fgt2

In [56]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.942398,0.835284,0.79187,NaN,NaN,NaN,NaN
t2,0.894478,0.648886,0.543959,NaN,NaN,NaN,NaN
t3,0.903816,0.685981,0.593747,NaN,NaN,NaN,NaN
t4,0.931637,0.758669,0.686761,NaN,NaN,NaN,NaN


## Calculo del índice de desigualdad de Atkinson

Vamos a calcular el índice de desigualdad de atkinson con un parámetro $\epsilon$ de aversión a la desigualdad y un $\mu$ que es igual a la media de los ingresos individuales, con la siguiente fórmula.

$\begin{equation}A = 1-\frac{1}{\mu}\left(\frac{1}{N}\sum_{i=1}^{N}y^{1-\epsilon}\right)^{1/(1-\epsilon)}\end{equation}$

Donde $y_{i}$ es el ingreso individual y $\mu$ es el ingreso medio

In [57]:
for t in [1, 2, 3, 4]:
    # Filtramos para cada trimestre
    df_temp = data.loc[data[f'ingr_t_t{t}'] >= 0].copy()

    # Suma total de los factores de expansión para el trimestre
    peso_total = df_temp['fexp'].sum()
    
    # Ingreso promedio ponderado
    mu = (df_temp[f'ingr_t_t{t}'] * df_temp['fexp']).sum() / peso_total

    # Calculamos el índice A para epsilon 0.25, 0.5 y 0.75 utilizando los pesos
    indices = {}
    for i in [0.25, 0.5, 0.75]:
        A_i = ((df_temp[f'ingr_t_t{t}']**(1-i) * df_temp['fexp']).sum() / peso_total)**(1/(1-i))
        indices[i] = A_i

    # Ratio de pobreza con el índice total (aplicando la fórmula)
    a25 = 1 - 1/mu * indices[0.25]
    a50 = 1 - 1/mu * indices[0.5]
    a75 = 1 - 1/mu * indices[0.75]

    # Guardamos los resultados
    datos_final.loc[f't{t}', 'a25'] = a25
    datos_final.loc[f't{t}', 'a50'] = a50
    datos_final.loc[f't{t}', 'a75'] = a75


In [58]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.942398,0.835284,0.79187,0.3932,0.750993,0.978338,NaN
t2,0.894478,0.648886,0.543959,0.367743,0.591237,0.837121,NaN
t3,0.903816,0.685981,0.593747,0.431555,0.668384,0.891208,NaN
t4,0.931637,0.758669,0.686761,0.290373,0.606728,0.917329,NaN


Guardamos el ingreso promedio

In [59]:
for t in [1, 2, 3, 4]:
    df_temp = data.loc[data[f'ingr_t_t{t}'] >= 0].copy()
    
    # Calcula la suma total de los factores de expansión
    peso_total = df_temp['fexp'].sum()
    
    # Calcula el ingreso promedio ponderado
    media_ponderada = (df_temp[f'ingr_t_t{t}'] * df_temp['fexp']).sum() / peso_total
    
    datos_final.loc[f't{t}', 'ingreso_promedio'] = media_ponderada

In [60]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.942398,0.835284,0.79187,0.3932,0.750993,0.978338,16.690824
t2,0.894478,0.648886,0.543959,0.367743,0.591237,0.837121,47.980865
t3,0.903816,0.685981,0.593747,0.431555,0.668384,0.891208,49.96358
t4,0.931637,0.758669,0.686761,0.290373,0.606728,0.917329,23.83671


In [61]:
datos_final.to_csv('datos_final.csv')